# 재하봇 호출어 **v5** 학습 — 합성은 끝났고, 여기선 augment~train~export 만 한다

**v4 와 무엇이 다른가: 증강 격자에 '낮은 피치' 칸을 넣었다. 그것 하나뿐이다.**
TTS·원본 음성·개수·학습 설정은 v4 와 같다 — 그래야 결과 차이를 격자 탓으로 좁힐 수 있다.

왜: v4 실기(2026-08-12, 실음성 44건)에서 **성인 남성 목소리가 통째로 약했다.**
또박또박 천천히 불러도 통과 50%(점수 중앙 0.223). 녹음의 피치만 올려 재채점하면:

| 조건 | 원본 | +15% | +35% |
|---|---|---|---|
| 또박또박 | 0.223 | 0.351 | **0.814** |
| 보통 | 0.179 | **0.496** | 0.481 |
| 빠르게 | 0.080 | **0.320** | 0.125 |
| 흘려서 | 0.041 | 0.092 | 0.080 |

음량은 원인이 아니다(학습셋 음량으로 정규화해도 −0.010). 배경소음·SNR·F0 상관도 0 근처.
분포 문제다 — 학습 긍정 F0 중앙 **165Hz**(5~95% 88~229)인데 사용자는 **104~116Hz**.
게다가 v4 는 피치를 올리는 쪽으로만 증강해 무게중심을 더 위로 밀어 놨다.

➡️ 녹음을 +15%/+35% 올려 좋아졌다는 건, 학습 쪽에서 1/1.15=0.87 · 1/1.35=0.74 로
**낮춘** 데이터를 넣는 것과 같은 말이다. 그래서 **0.85 · 0.75** 칸을 추가했다.

**격자 7칸 → 11칸** (클립당 3칸 무작위는 그대로 → 총 개수 불변)

| 시간＼피치 | 0.75 | 0.85 | 1.0 | 1.15 | 1.35 |
|---|---|---|---|---|---|
| 1.0 | **v5** | **v5** | (원본) | v4 | v4 |
| 1.15 | | **v5** | v4 | | v4 |
| 1.25 | | | | v4(1.25) | |
| 1.35 | | **v5** | v4 | v4 | |

**가져오는 데이터** (`jaehabot_v5_wav.tar.gz`, 약 340MB) — 개수는 v3·v4 와 동일

| 폴더 | 개수 | 비고 |
|---|---|---|
| positive_train | 7,276 | 검수 통과 원본 1819 × 4 |
| positive_test | 1,816 | 시드를 분리해 만든 별도 셋 |
| negative_train | 8,000 | 함정 문구 17종 |
| negative_test | 2,000 | |

> ⚠️ **부정에도 긍정과 똑같은 증강이 들어가 있다.** 긍정에만 걸면 모델이
> '그 변형 = 호출어'라는 지름길을 배운다. v3 때 이건 챙겼는데 **축 독립성**을 놓쳤다.

> ⚠️ **흘려서 부르는 경우(축약 발음)는 v5 도 못 고친다.** 위 표에서 유일하게
> 피치로 안 오르는 조건이다. 신호처리로는 `재하봇`에서 `잰봇`을 만들 수 없다 —
> 축약형 문구 추가나 실음성 재학습이 필요하고, 그건 v5 결과를 보고 정한다.

---
**실행 전:** 런타임 → 런타임 유형 변경 → **GPU**
**준비물:** Drive 의 `jaeha_wake_backup/` 안에 `jaehabot_v5_wav.tar.gz` 를 올려둘 것

In [ ]:
!nvidia-smi
import sys; print('python', sys.version)

In [ ]:
# ── 경로·토큰 ──────────────────────────────────────────────
import os
from google.colab import drive, userdata

drive.mount('/content/drive')
os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')   # 배경음(MUSAN) 다운로드 rate limit 회피

MODEL_NAME = 'jaehabot_v5'          # v1·v2 산출물을 덮어쓰지 않는다
WORK   = '/content/jaeha_wake'
DATA   = os.path.join(WORK, 'data')
OUTDIR = os.path.join(WORK, 'output')
OUT    = os.path.join(OUTDIR, MODEL_NAME)
BACKUP = '/content/drive/MyDrive/jaeha_wake_backup'
DATA_TAR = os.path.join(BACKUP, 'jaeha_wake_data.tar')
OUR_TAR  = os.path.join(BACKUP, 'jaehabot_v5_wav.tar.gz')   # 젯슨에서 만든 우리 데이터

for d in (WORK, DATA, OUTDIR, BACKUP):
    os.makedirs(d, exist_ok=True)
os.makedirs(os.path.join(WORK, 'configs'), exist_ok=True)
os.chdir(WORK)

print('우리 데이터:', OUR_TAR, '->', '있음' if os.path.exists(OUR_TAR) else '🔴 없음! Drive 에 올렸는지 확인')
!df -h /content | cat

In [ ]:
# ── 설치 (이미 깔려 있으면 건너뜀) ──────────────────────────
import importlib.util

if importlib.util.find_spec('livekit') is None:
    !apt-get -qq install -y espeak-ng libsndfile1 ffmpeg sox
    !pip -q install 'livekit-wakeword[train,eval,export,voxcpm]'
else:
    print('livekit-wakeword 이미 설치됨 — 건너뜀')

import torch
print('torch', torch.__version__, '| cuda:', torch.cuda.is_available(),
      '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU!')

In [ ]:
# ── 진행이 보이는 실행 헬퍼 (셸매직은 출력이 버퍼링돼 죽었는지 도는지 알 수 없다) ──
import subprocess, sys, time
from IPython.display import clear_output

def run_stream(*args, every=15.0, keep_tail=8):
    p = subprocess.Popen(list(args), stdout=subprocess.PIPE,
                         stderr=subprocess.STDOUT, text=True, bufsize=1)
    tail, n, last = [], 0, 0.0
    for line in p.stdout:
        tail.append(line.rstrip())
        if len(tail) > keep_tail: tail.pop(0)
        n += 1
        if time.time() - last >= every:
            clear_output(wait=True)
            print(f'[{n}줄 진행중] {" ".join(args[:2])}')
            print('\n'.join(tail)); sys.stdout.flush()
            last = time.time()
    p.wait()
    clear_output(wait=True)
    print('\n'.join(tail))
    print(f'\n--- exit code: {p.returncode} (총 {n}줄) ---')
    if p.returncode != 0:
        raise RuntimeError(f'{args[0]} 실패(exit {p.returncode}) — 위 로그 확인')

## 1. config

v2 설정을 그대로 쓰되 세 가지만 바꾼다.

- `model_name` → `jaehabot_v5` (v1·v2 산출물 보존)
- `target_phrases` → 실제로 합성한 3개(`재하봇 / 재하봇아 / 재하봇이`)로 맞춤
- **`n_samples` 50** — voxcpm 으로 음성을 만들 생각이 없다. generate 를 돌리는
  목적은 오직 **배경음(MUSAN) 폴더를 만드는 것**이고, 거기서 딸려 나오는
  voxcpm 긍정·부정은 다음 셀에서 우리 데이터로 통째로 덮어쓴다.

In [ ]:
import yaml

BASE_YAML = r'''
model_name: jaehabot_v2
# ============================================================================
# v2 변경점 (2026-08-05) — 빠른 발화 미인식 수정
#
# 문제(2026-08-02 실기): "천천히 부르면 잘 되는데 빠르게 부르면 인식 안 됨"
# 원인: voice_design_prompts 6개에 **속도 묘사가 하나도 없었다.** 공식 영어 base 에는
#       moderate/slightly faster/quick conversational/slower pace 가 골고루 있었는데
#       한국어로 교체하면서 속도 축을 통째로 빠뜨렸다. 결과적으로 긍정 샘플이 전부
#       비슷한 보통~느린 속도가 되어 빠른 발화가 학습 분포 밖에 놓였다.
# ⚠️ length_scales 는 **Piper 전용 키라 voxcpm 백엔드에선 무시된다.**
#    voxcpm 에서 속도를 바꾸는 유일한 수단은 프롬프트 텍스트뿐이다.
#
# 수정: 음색·나이 축 × 속도 축으로 프롬프트를 재구성(6개 → 16개).
#       빠름 7 / 보통 6 / 느림 3. 빠른 쪽에 가중을 뒀다 — 그게 지금 비어 있는 구간이고,
#       느림은 v1 이 사실상 그 구간이라 이미 잘 되므로 회귀 방지 최소분만 남겼다.
# ============================================================================
# 🔴 '재하봇아' 강화 (2026-08-05): 호격 '~아'는 연음으로 [재하보사] 로 실현된다.
#    voxcpm 이 한국어 연음을 제대로 렌더링한다는 보장이 없어 **발음형을 병기**한다.
#    근거: configs/model_paths.yaml 의 stt.aliases 에 제아부사/재아부사 -> 재하봇 이
#    등록돼 있다 = 실제 사람이 부를 때 그렇게 들린다는 뜻.
#
#    v1 의 5개에서 '재하 봇'·'재하보'·'재하봇아아'(잘 안 쓰는 변형)를 빼고
#    호격형과 그 발음 변형에 집중해 7개로 재구성했다. 문구가 줄면 문구당 샘플이
#    두꺼워진다(3000/7 = 약 428개, v1 은 2000/5 = 400개).
target_phrases:
- 재하봇
- 제하봇        # 첫 음절이 '제'로 들리는 변형 (aliases 의 제아부사 계열과 같은 갈래)
- 재하봇아      # 호격 표기형
- 재하보사      # '재하봇아' 연음 실현형 — TTS 가 연음을 안 살릴 때의 보험
- 재하부사      # 실기 오인식형 (aliases: 제아부사/재아부사)
- 재하봇이      # 주격 표기형
- 재하보시      # '재하봇이' 연음 실현형
tts_backend: voxcpm
# 프롬프트가 6 -> 16 으로 늘어 프롬프트당 샘플 수가 1/3 수준으로 얇아진다.
# 2000 을 유지하면 프롬프트당 125개뿐 → 3000 으로 올려 187개를 확보한다.
# ⚠️ 생성 시간이 비례해 늘어난다(voxcpm 은 순차 생성, 2000개 ≈ 2.5~3시간 → 3000개 ≈ 4~4.5시간).
#    시간이 부족하면 2000 으로 되돌리되, 프롬프트 수를 12개로 줄이는 쪽을 먼저 검토할 것.
n_samples: 3000
n_samples_val: 750
n_background_samples: 3000
n_background_samples_val: 750
tts_batch_size: 256     # ⚠️ voxcpm_backend.py 가 `del batch_size` 로 무시한다(순차 생성). 무해해서 남겨둠.
custom_negative_phrases:
- 자동차
- 해바라기
- 지하철
- 재밌다
- 재미있어
- 재현이
- 로봇
- 보트
- 하마
- 자전거
- 엄마
- 아빠
- 이거 뭐야
- 밥 먹자
- 같이 놀자
- 안녕
- 싫어
noise_scales:
- 0.98
noise_scale_ws:
- 0.98
length_scales:          # ⚠️ Piper 전용 — voxcpm 에선 무시됨. 속도는 프롬프트로만 제어된다.
- 0.75
- 1.0
- 1.25
slerp_weights:
- 0.2
- 0.35
- 0.5
- 0.65
- 0.8
piper_tts:
  checkpoint_relpath: piper/en-us-libritts-high.pt
voxcpm_tts:
  model_id: openbmb/VoxCPM2
  model_cache_relpath: voxcpm/VoxCPM2
  local_model_path: null
  load_denoiser: false
  cfg_values:
  - 1.5
  - 2.0
  - 2.5
  - 3.0
  inference_timesteps_list:   # 기본 [8,10,12] 에서 낮춤 — 생성 속도의 유일한 실질 레버
  - 4
  - 6
  - 8
  # ⚠️ 반드시 voxcpm_tts 아래에 중첩할 것. 최상위에 쓰면 조용히 무시되고 원본 영어 33종이 쓰인다.
  voice_design_prompts:
  # --- 빠름 (7개) — v1 에 없던 구간. 실기 실패의 원인이라 가장 두껍게 ---
  - 빠르게 부르는 한국인 어린 여자아이
  - 급하게 외치듯 부르는 한국인 남자아이
  - 신이 나서 빠르게 말하는 어린아이
  - 다급하게 부르는 어린아이
  - 빠른 말투로 또렷하게 말하는 한국인 성인 여성
  - 빠르게 툭 내뱉듯 부르는 어린아이
  - 숨차게 빠르게 부르는 한국인 아이
  # --- 보통 (6개) — v1 프롬프트를 속도 표현만 붙여 승계 ---
  - 보통 속도로 밝고 높게 말하는 한국인 어린 여자아이
  - 보통 속도로 해맑게 말하는 한국인 남자아이
  - 보통 속도로 또렷하게 말하는 한국인 성인 여성
  - 보통 속도로 차분하게 말하는 한국인 성인 남성
  - 보통 속도로 조금 웅얼거리며 말하는 어린아이
  - 신나서 크게 부르는 한국인 어린아이
  # --- 느림 (3개) — v1 이 사실상 이 구간이라 이미 잘 된다. 회귀 방지 최소분만 유지 ---
  - 아주 천천히 또박또박 부르는 한국인 어린아이
  - 천천히 조심스럽게 부르는 한국인 여자아이
  - 느릿하게 웅얼거리는 어린아이
data_dir: /content/jaeha_wake/data
output_dir: /content/jaeha_wake/output
augmentation:
  clip_duration: 2.0
  batch_size: 64
  rounds: 3
  background_paths:
  - ./data/backgrounds
  rir_paths:
  - ./data/rirs
model:
  model_type: conv_attention
  model_size: medium
steps: 50000
learning_rate: 0.0001
weight_decay: 0.01
label_smoothing: 0.05
max_negative_weight: 3000
target_fp_per_hour: 0.1
batch_n_per_class:
  positive: 50
  adversarial_negative: 50
  ACAV100M_sample: 1024
  background_noise: 50
'''

cfg = yaml.safe_load(BASE_YAML)
cfg['model_name'] = MODEL_NAME
cfg['data_dir'], cfg['output_dir'] = DATA, OUTDIR

# 실제로 합성한 문구와 맞춘다(비어휘 변형은 Supertonic 에서도 안 쓴다)
cfg['target_phrases'] = ['재하봇', '재하봇아', '재하봇이']

# voxcpm 합성은 최소로 — 배경음 폴더만 얻으면 된다
cfg['n_samples'], cfg['n_samples_val'] = 50, 10
cfg['n_background_samples'], cfg['n_background_samples_val'] = 3000, 750

CFG_PATH = 'configs/jaeha_v5.yaml'
yaml.safe_dump(cfg, open(CFG_PATH, 'w', encoding='utf-8'),
               allow_unicode=True, sort_keys=False)
print('저장:', CFG_PATH)
print('  model_name  :', cfg['model_name'])
print('  n_samples   :', cfg['n_samples'], '(배경음용 최소값 — 우리 데이터로 덮어쓴다)')
print('  augment     :', cfg['augmentation']['rounds'], '라운드')
print('  steps       :', cfg['steps'])

In [ ]:
# ── 데이터 준비 (있으면 건너뛴다) ───────────────────────────
have_feat = os.path.isdir(os.path.join(DATA, 'features'))
have_bg   = os.path.isdir(os.path.join(DATA, 'backgrounds'))

if have_feat and have_bg:
    print('데이터 이미 있음 — 다운로드 건너뜀')
elif os.path.exists(DATA_TAR):
    print('Drive 캐시에서 복원:', DATA_TAR)
    run_stream('tar', '-C', WORK, '-xf', DATA_TAR)
else:
    print('데이터 없음 → setup 실행 (ACAV100M 특징 16GB 등, 수십 분)')
    run_stream('livekit-wakeword', 'setup', '--config', CFG_PATH)

!du -sh {DATA}/* 2>/dev/null | cat

## 2. generate — **배경음 폴더를 만들려고** 돌린다 (10분 안팎)

여기서 나오는 `positive_*` · `negative_*` 는 voxcpm 산이라 쓸 물건이 아니다.
다음 셀에서 통째로 지우고 우리 것으로 바꾼다.

In [ ]:
run_stream('livekit-wakeword', 'generate', CFG_PATH, every=20, keep_tail=5)

import glob
for d in sorted(glob.glob(OUT + '/*')):
    if os.path.isdir(d):
        print(f'{len(glob.glob(d + "/*.wav")):7d}  {os.path.basename(d)}')

## 3. ★ 우리 데이터로 교체 — 여기가 이 노트북의 핵심

voxcpm 이 만든 긍정·부정을 **지우고** 젯슨에서 검수까지 끝낸 것을 푼다.
배경음(`background_*`)은 실제 소음 데이터셋이라 그대로 둔다.

In [ ]:
import shutil, glob

# 1) voxcpm 산 긍정·부정 제거 (남겨두면 검수 안 된 음성이 학습에 섞인다)
for sub in ['positive_train', 'positive_test', 'negative_train', 'negative_test']:
    p = os.path.join(OUT, sub)
    n = len(glob.glob(p + '/*.wav'))
    shutil.rmtree(p, ignore_errors=True)
    print(f'제거 {sub}: {n}개')

# 2) 우리 데이터 풀기 (tar 안이 jaehabot_v5/... 구조라 OUTDIR 에 푼다)
assert os.path.exists(OUR_TAR), f'없음: {OUR_TAR} — Drive 에 올렸는지 확인'
print('\n복원 중(수 분):', OUR_TAR)
run_stream('tar', '-C', OUTDIR, '-xzf', OUR_TAR)

print()
EXPECT = {'positive_train': 7276, 'positive_test': 1816,
          'negative_train': 8000, 'negative_test': 2000}
ok = True
for sub, want in EXPECT.items():
    got = len(glob.glob(os.path.join(OUT, sub, 'clip_*.wav')))
    mark = 'OK ' if got == want else '🔴'
    ok = ok and got == want
    print(f'{mark} {sub:<16} {got:6d} (기대 {want})')
for d in sorted(glob.glob(OUT + '/background*')):
    print(f'    {os.path.basename(d):<16} {len(glob.glob(d + "/*.wav")):6d}')
assert ok, '개수가 다르다 — tar 가 깨졌거나 덜 풀렸다'
print('\n교체 완료')

### 3-2. 들어보고 넘어간다

v2 의 실패는 **아무도 안 들어본 것**이었다. 4시간짜리 학습을 걸기 전에 확인한다.

In [ ]:
import random, soundfile as sf
from IPython.display import Audio, display

for sub in ['positive_train', 'negative_train']:
    clips = sorted(glob.glob(os.path.join(OUT, sub, 'clip_*.wav')))
    print(f'=== {sub} ({len(clips)}개) 중 3개 ===')
    for c in random.Random(0).sample(clips, 3):
        y, sr = sf.read(c)
        print(' ', os.path.basename(c), f'{len(y)/sr:.2f}s')
        display(Audio(y, rate=sr))

## 4. augment — 소음·잔향·EQ (20~40분)

내장 증강은 EQ·왜곡·RIR잔향·배경믹싱이다. 피치/속도는 없지만,
우리는 그 두 축을 젯슨에서 이미 넣었다(긍정·부정 대칭).

In [ ]:
run_stream('livekit-wakeword', 'augment', CFG_PATH, every=30)

In [ ]:
# augment 검증 — _rN 이 0이면 학습 데이터가 비어 있는 것이므로 train 금지
for sub in ['positive_train', 'negative_train']:
    d = os.path.join(OUT, sub)
    tot = len(glob.glob(d + '/*.wav'))
    aug = len(glob.glob(d + '/*_r*.wav'))
    print(f'{sub:<16} 전체 {tot:6d} / 증강본 {aug:6d}')
    assert aug > 0, f'{sub} 증강본 없음 — 이 상태로 train 금지'
print('OK')

### 4-2. 💾 백업 (권장)

여기까지가 젯슨 50분 + Colab augment 결과다. 런타임이 끊기면 다시 만들어야 한다.
(우리 원본 tar 은 Drive 에 있으니 최악이어도 3번부터 다시 하면 된다.)

In [ ]:
DO_BACKUP = True
TAR = os.path.join(BACKUP, f'{MODEL_NAME}_augmented.tar')
if DO_BACKUP:
    run_stream('tar', '-C', OUTDIR, '-cf', TAR, MODEL_NAME)
    !ls -lh {TAR} | cat
else:
    print('건너뜀')

## 5. train (수 시간)

In [ ]:
run_stream('livekit-wakeword', 'train', CFG_PATH, every=60)

## 6. export — ONNX

⚠️ `--quantize`(int8)는 shape inference 에러로 실패한다. 젯슨에 int8 은 불필요.

In [ ]:
run_stream('livekit-wakeword', 'export', CFG_PATH)

## 7. 공식 평가

실제 음성 ~18시간(ACAV100M 검증셋)을 부정으로 써서 지표를 낸다.
**v1 기준선: AUT 0.0302 / FPPH 0.22 / Recall 40.1% @ thr 0.50**

In [ ]:
run_stream('livekit-wakeword', 'eval', CFG_PATH)

## 8. ★ threshold 스윕 — 젯슨에 넣을 값을 고른다

**v1 실측 (비교 기준):**

| thr | recall | FPPH |
|---|---|---|
| 0.30 | 49.3% | 0.65 |
| 0.20 | 55.1% | 1.20 |
| **0.15** | **58.2%** | **2.18** ← 젯슨 배포값 |
| 0.10 | 63.0% | 4.24 |

**판정:** v5 가 **FPPH 2.18 근처에서 recall 58.2% 를 넘으면** 개선이다.

⚠️ 단, v1 과 v5 는 **시험지가 다르다**(positive_test 를 각자 만들었다).
v5 의 테스트셋에는 v1 이 못 맞히는 빠른 발화가 들어 있어 v5 에 더 어렵다.
그래서 이 표는 방향만 보고, **최종 판정은 젯슨 실기**로 한다.

In [ ]:
import numpy as np, onnxruntime as ort

sess = ort.InferenceSession(OUT + f'/{MODEL_NAME}.onnx', providers=['CPUExecutionProvider'])
iname = sess.get_inputs()[0].name

def load_feats(path):
    a = np.load(path, mmap_mode='r')
    if a.ndim == 2:
        n = (len(a) // 16) * 16
        a = np.asarray(a[:n]).reshape(-1, 16, 96)
    return a

def scores(path, bs=4096):
    a = load_feats(path); out = []
    for i in range(0, len(a), bs):
        x = np.asarray(a[i:i+bs], dtype=np.float32)
        out.append(sess.run(None, {iname: x})[0].reshape(-1))
    return np.concatenate(out)

ps = scores(OUT + '/positive_features_test.npy')
ns = np.concatenate([scores(f) for f in [
    OUT + '/negative_features_test.npy',
    OUT + '/background_noise_features_test.npy',
    DATA + '/features/validation_set_features.npy']])
hours = len(ns) * 2.0 / 3600
print(f'긍정 {len(ps)} / 부정 {len(ns)} ({hours:.2f}시간)\n')

V1 = {0.30: (49.3, 0.65), 0.20: (55.1, 1.20), 0.15: (58.2, 2.18), 0.10: (63.0, 4.24)}
print(f'{"thr":>6} {"recall":>8} {"FPPH":>9}   {"v1":>8} {"차이":>8}')
for t in [0.5,0.4,0.3,0.25,0.2,0.15,0.12,0.1,0.08,0.06,0.05,0.04,0.03,0.02]:
    r, f = (ps>=t).mean()*100, (ns>=t).sum()/hours
    if t in V1:
        print(f'{t:6.2f} {r:7.1f}% {f:9.2f}   {V1[t][0]:7.1f}% {r-V1[t][0]:+7.1f}%p')
    else:
        print(f'{t:6.2f} {r:7.1f}% {f:9.2f}')

## 9. 젯슨 배포용 zip (추론엔 ONNX 3개가 필요하다)

In [ ]:
import importlib.util
EXP = '/content/jaeha_wake_export_v5'
shutil.rmtree(EXP, ignore_errors=True); os.makedirs(EXP)

for f in glob.glob(OUT + '/**/*.onnx', recursive=True): shutil.copy(f, EXP)
for f in glob.glob(OUT + '/**/*.pt', recursive=True):   shutil.copy(f, EXP)

spec = importlib.util.find_spec('livekit.wakeword')
pkg = os.path.dirname(spec.origin) if spec and spec.origin else ''
for name in ['melspectrogram.onnx', 'embedding_model.onnx']:
    hits = glob.glob(pkg + '/**/' + name, recursive=True)
    print(name, '->', hits[:1])
    if hits: shutil.copy(hits[0], EXP)
    else:    print(f'  [!] {name} 못 찾음 — 젯슨 추론에 필요하니 수동 확인')

shutil.copy(CFG_PATH, EXP)
zip_local = shutil.make_archive('/content/jaeha_wake_export_v5', 'zip', EXP)
shutil.copy(zip_local, BACKUP)
print('\n묶음:', sorted(os.listdir(EXP)))
print('Drive 사본:', os.path.join(BACKUP, os.path.basename(zip_local)))

## 10. 다음 (젯슨)

1. zip 을 `~/jaeha_bot/models/wake/v5/` 에 푼다 — **v1 파일을 덮어쓰지 말 것**(롤백용).
2. `configs/model_paths.yaml`:
   ```yaml
   wake:
     onnx:
       classifier: models/wake/v5/jaehabot_v5.onnx
       threshold: <8번 스윕에서 고른 값>
       trigger_frames: 1   # v1 에서 빠른 발화 때문에 2->1 로 완화했던 값.
                           # v5 는 빠른 발화를 학습했으니 2 로 되돌려도 되는지 재검토
   ```
3. **실기 확인이 진짜 판정이다.** 천천히/보통/빠르게 각 10회씩 불러 깨어난 비율을 센다.
   지금까지 recall 은 전부 *합성음 시험지* 점수라 실제 성능이 아니다.
4. 헛깨움이 늘면 threshold 를 올리고, 빠른 호출을 놓치면 낮춘다.
   **못 깨우는 게 헛깨움보다 치명적**이라는 게 지금까지의 판단 기준이었다.

> 🔴 이번에도 안 되면 다음 수는 **실음성**이다. 젯슨 ReSpeaker 로 재하 30~50회,
> 부모 20~30회, 거실 소음 1시간을 녹음해 분류기를 다시 학습한다.
> 임베딩(`embedding_model.onnx`)은 고정이고 분류기만 바꾸면 되므로 적은 데이터로도 효과가 크다.

## 9-4. 🔴 격자 칸별 재현율 — 평균은 실패를 숨긴다

위 threshold 스윕은 **전체 평균**이라 특정 조건만 못 잡아도 숫자가 좋게 나온다.
v3 가 정확히 그렇게 실패했다(오프라인 91% / 실기에서 빠른 발화 놓침).

**v4 는 이 검사를 20%p 기준으로 '통과'했는데도 실기에서 실패했다.** 최악 칸이
19%p 라 1%p 차이로 빠져나갔고, 게다가 문제는 한 칸이 튄 게 아니라
**한 축이 통째로 기운 것**이었다(시간 1.35 인 칸 둘 다 65~67%).
그래서 아래 판정은 최악 '칸' 하나가 아니라 **축별 평균**까지 같이 본다.

이번에 꼭 볼 칸:
- **피치 0.85 / 0.75** = 성인 남성 음역 ← v5 가 노린 칸. 여기가 낮으면 v5 는 실패다
- **시간 1.35** = 빠른 발화 ← v4 가 약했던 축(65~67%)
- **시간 1.0 / 피치 1.35** = 아이 목소리 ← 원래 잘 되던 칸. **떨어지면 안 된다**

**어느 칸이든 전체 평균보다 크게 낮으면 그 조건은 실기에서 실패한다.**

In [ ]:
# 내보낸 ONNX 3종을 런타임과 같은 순서로 태워 원본 wav 를 채점한다.
# (app/wake_onnx.py 와 같은 계산 — 멜은 int16 스케일 입력 + x/10+2 정규화)
import json, glob, numpy as np, soundfile as sf, onnxruntime as ort
from collections import defaultdict

P = ['CPUExecutionProvider']
mel_s = ort.InferenceSession(EXP + '/melspectrogram.onnx', providers=P)
emb_s = ort.InferenceSession(EXP + '/embedding_model.onnx', providers=P)
cls_s = ort.InferenceSession(EXP + f'/{MODEL_NAME}.onnx', providers=P)
MEL_WINDOW, EMB_WINDOW, MEL_BANDS, EMB_DIM, FRAME = 76, 16, 32, 96, 1280

def best_score(y):
    pad = (np.random.randn(32000) * 1e-4).astype(np.float32)
    s = np.concatenate([pad, y.astype(np.float32), pad[:8000]])
    mel_buf, emb_buf, best = [], [], 0.0
    for i in range(0, len(s) - FRAME, FRAME):
        a = (s[i:i+FRAME].reshape(1, -1) * 32767).astype(np.float32)
        m = np.squeeze(mel_s.run(None, {mel_s.get_inputs()[0].name: a})[0])
        m = m.reshape(-1, MEL_BANDS) / 10.0 + 2.0
        mel_buf.extend(m.astype(np.float32))
        mel_buf = mel_buf[-MEL_WINDOW:]
        if len(mel_buf) < MEL_WINDOW:
            continue
        w = np.stack(mel_buf)[None, :, :, None].astype(np.float32)
        e = np.squeeze(emb_s.run(None, {emb_s.get_inputs()[0].name: w})[0])
        emb_buf.append(e.reshape(EMB_DIM).astype(np.float32))
        emb_buf = emb_buf[-EMB_WINDOW:]
        if len(emb_buf) < EMB_WINDOW:
            continue
        f = np.stack(emb_buf)[None, :, :].astype(np.float32)
        best = max(best, float(np.squeeze(
            cls_s.run(None, {cls_s.get_inputs()[0].name: f})[0])))
    return best

man = {}
with open(OUT + '/positive_test/manifest.jsonl', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            r = json.loads(line); man[r['clip']] = r

cells = defaultdict(list)
for nm, r in man.items():
    cells[(r.get('rate', 1.0), r.get('pitch', 1.0))].append(nm)

rng = np.random.RandomState(0)
THRS = [0.30, 0.25, 0.20, 0.15]
print(f"{'시간':>5} {'피치':>5} {'n':>4} " + ' '.join(f'{t:>7.2f}' for t in THRS))
print('-' * 46)
allsc = []
for key in sorted(cells):
    names = cells[key]
    pick = [names[i] for i in rng.choice(len(names), min(60, len(names)), replace=False)]
    sc = []
    for nm in pick:
        y, _ = sf.read(OUT + '/positive_test/' + nm, dtype='float32')
        sc.append(best_score(y.mean(axis=1) if y.ndim > 1 else y))
    sc = np.array(sc); allsc.append((key, sc))
    print(f'{key[0]:>5} {key[1]:>5} {len(sc):>4} ' +
          ' '.join(f'{(sc>=t).mean()*100:>6.0f}%' for t in THRS))

base = np.concatenate([s for _, s in allsc])
print(f"\n{'전체':>11} {len(base):>4} " +
      ' '.join(f'{(base>=t).mean()*100:>6.0f}%' for t in THRS))

# ── 판정 ──────────────────────────────────────────────────────────────
# v4 의 교훈: '최악 칸 하나'만 보면 축이 통째로 기운 걸 놓친다. 칸·축 둘 다 본다.
B = (base >= 0.25).mean()
rows = sorted(((k, (s >= 0.25).mean()) for k, s in allsc), key=lambda kv: kv[1])
print('\n낮은 칸 3개 (thr 0.25)')
for k, v in rows[:3]:
    print(f'   시간 {k[0]:<5} 피치 {k[1]:<5} {v*100:>3.0f}%  (전체보다 {(B-v)*100:>2.0f}%p 낮음)')

print('\n축별 평균 (thr 0.25)')
bad_axis = []
for name, idx in (('시간', 0), ('피치', 1)):
    vals = {}
    for k, s in allsc:
        vals.setdefault(k[idx], []).append((s >= 0.25).mean())
    for v in sorted(vals):
        m = float(np.mean(vals[v]))
        flag = '  🔴' if B - m > 0.15 else ''
        if flag:
            bad_axis.append(f'{name} {v}')
        print(f'   {name} {v:<5} {m*100:>3.0f}%{flag}')

worst_k, worst_v = rows[0]
gap = B - worst_v
print(f'\n최악 칸: 시간 {worst_k[0]} 피치 {worst_k[1]} -> {worst_v*100:.0f}% '
      f'(전체 {B*100:.0f}% 보다 {gap*100:.0f}%p 낮음)')
if gap > 0.20 or bad_axis:
    print('🔴 배포하지 말 것. ' +
          (f'기운 축: {", ".join(bad_axis)}. ' if bad_axis else '') +
          '학습 데이터를 고쳐야 한다.')
else:
    print('✅ 칸·축 모두 큰 쏠림이 없다.')
print('\n⚠️ 이 수치는 합성음 자기채점이다. 최종 판정은 실기 —'
      ' tools/record_wake_real.py 로 조건당 10회 이상.')